# study of the impact of depth uncertainty (ycen shift) on strain refinement results

ZrO2 case

In [ ]:
#%matplotlib inline
%matplotlib notebook

#import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#pd.__version__

import time,copy,os

import LaueTools.dict_LaueTools as dictLT
import LaueTools.indexingImageMatching as IMM
#import LaueTools.findorient as FO
import LaueTools.indexingSpotsSet as ISS

import LaueTools.generaltools as GT
import LaueTools.IOLaueTools as IOLT
import LaueTools.lauecore as LT
#import LaueTools.LaueGeometry as LTGeo


# third party LaueTools import
import LaueTools.CrystalParameters as CP

#import LaueTools.readmccd as RMCCD

from scipy.spatial.transform import Rotation as R

In [ ]:
# simulation
nbUBs=10
Rs = R.random(nbUBs,random_state=1234)
quatlist = Rs.as_quat()
UBlist = Rs.as_matrix()

In [ ]:
# For ZrO2
key_material ='ZrO2'
a0, b0, c0, alpha0, beta0, gamma0 = dictLT.dict_Materials[key_material][1]
Gstar = CP.Gstar_from_directlatticeparams(a0, b0, c0, alpha0, beta0, gamma0)
dictLT.dict_Materials[key_material][1]
rules = dictLT.dict_Materials[key_material][-1]
rules
#detectorparameters = [77.088,1012.45,1049.92,0.423,0.172]
detectorparameters = [79.000,975.00,941.00,0,0]

pixelsize = 0.0734
framedim = (2018,2016)

CCDCalibdict = {'dd': detectorparameters[0],
            'xcen': detectorparameters[1],
            'ycen': detectorparameters[2], 'xbet': detectorparameters[3],
            'xgam': detectorparameters[4], 'xpixelsize': pixelsize, 'ypixelsize': pixelsize,
            'CCDLabel': 'sCMOS', 'framedim': framedim, 'detectordiameter': 165,
            'kf_direction': 'Z>0', 'pixelsize': pixelsize}

In [ ]:
mat_idx = 6

UBmatrix = UBlist[mat_idx]
grain = CP.Prepare_Grain(key_material, UBmatrix)

sample_depth=0

detectorparameterscorrected = copy.copy(detectorparameters)


s_tth, s_chi, s_miller_ind, s_posx, s_posy, s_E= LT.SimulateLaue_full_np(grain, 5,22,detectorparameters,
                                                                         pixelsize=pixelsize,detectordiameter=300,
                                                                         dim=framedim,
                                                                         depth=10.,
                                                                        removeharmonics=1)

folder = '/home/micha/LaueProjects/DepthEffects'
prefixcorfilename = 'testZrO2_%d'%mat_idx
IOLT.writefile_cor(prefixcorfilename,s_tth, s_chi,s_posx, s_posy,1000./s_E, param=CCDCalibdict,
                  dirname_output = folder)

corfilename = prefixcorfilename + '.cor'

fullpathcorfile =os.path.join(folder,corfilename)

if 0:
    figlaue, axlaue= plt.subplots()
    axlaue.scatter(s_posx,s_posy, s=800./np.power(s_E,3), c='red')
    axlaue.set_xlim(0,2048)
    axlaue.set_ylim(2048,0)
    axlaue.set_aspect('equal')

In [ ]:
dict_loop = {'MATCHINGRATE_THRESHOLD_IAL': 100,
                   'MATCHINGRATE_ANGLE_TOL': 0.2,
                   'NBMAXPROBED': 6,
                   'central spots indices': [0,],
                   'AngleTolLUT': 0.5,
                   'UseIntensityWeights': False,
                   'nbSpotsToIndex':10000,
                   'list matching tol angles':[0.2,0.2,0.1],
                   'nlutmax':4,
                   'MinimumNumberMatches': 3,
                   'MinimumMatchingRate':3
                   }
grainindex=0
DataSet = ISS.spotsset()
    
DataSet.pixelsize = CCDCalibdict['xpixelsize']
DataSet.dim = CCDCalibdict['framedim']
DataSet.detectordiameter = CCDCalibdict['detectordiameter']
DataSet.kf_direction = CCDCalibdict['kf_direction']
DataSet.key_material = key_material
DataSet.emin = 5
DataSet.emax = 22

CheckFirstThisMatrix=UBmatrix
previousResults = 1, [CheckFirstThisMatrix], 50, 50

In [ ]:
initCCDCalibdict = copy.copy(CCDCalibdict)

sols=[]
ublist =[]
lats=[]
list_depth = [-100,-50,-25,-20,-15,-10,-5,0,5,10,15,20,25,50,100]
list_depth = [-50,0,50]
for dd in list_depth:
    grainindex=0
    DataSet = ISS.spotsset()
    DataSet.importdatafromfile(fullpathcorfile)
    DataSet.pixelsize = CCDCalibdict['xpixelsize']
    DataSet.dim = CCDCalibdict['framedim']
    DataSet.detectordiameter = CCDCalibdict['detectordiameter']
    DataSet.kf_direction = CCDCalibdict['kf_direction']
    DataSet.key_material = key_material
    DataSet.emin = 5
    DataSet.emax = 22
    #deltaycen = dd/1000./DataSet.pixelsize
    #DataSet.CCDcalibdict['ycen']=initCCDCalibdict['ycen']+deltaycen
    #DataSet.detectorparameters[2]=DataSet.CCDcalibdict['ycen']
    
    DataSet.IndexSpotsSet(fullpathcorfile, key_material, 5, 22, dict_loop, None,
                             use_file=1, # if 1, reimport data from file and reset also spots properties dictionary
                             IMM=False,LUT=None,n_LUT=dict_loop['nlutmax'],
                          angletol_list=dict_loop['list matching tol angles'],
                            nbGrainstoFind=1,
                          starting_grainindex=0,
                          MatchingRate_List=[1, 1, 1,1,1,1,1,1],
                            verbose=0, previousResults=previousResults,
                          corfilename=corfilename,
                         depth=dd)

    UBmatrixrefined = DataSet.dict_grain_matrix[0]
    lattice_parameter_direct_strain = CP.computeLatticeParameters_from_UB(
                                                            UBmatrixrefined, key_material, 'a',
                                                            dictmaterials=dictLT.dict_Materials)
    sols.append(np.ravel(DataSet.dict_grain_devstrain[0]))
    
    ublist.append(UBmatrixrefined)
    lats.append(lattice_parameter_direct_strain)

epsstrain=np.array(sols)
latparams = np.array(lats)

In [ ]:


fig, ax= plt.subplots()
ax.plot(list_depth,epsstrain[:,0],'o-', c='r')  # xx
ax.plot(list_depth,epsstrain[:,4],'o-', c='g')  # yy
ax.plot(list_depth,epsstrain[:,8],'o-', c='b')  # zz
ax.set_xlabel('normal to surface sample depth (µm)')
ax.set_ylabel('strain')

ax.grid(True)

figl, axl= plt.subplots()
axl.plot(list_depth,latparams[:,1],'o-', c='r')
axl.plot(list_depth,latparams[:,2],'o-', c='g')
axl.set_xlabel('normal to surface sample depth (µm)')
axl.set_ylabel('lattice parameter (Angstrom)')
axl.grid(True)

figl, axl= plt.subplots()
axl.plot(list_depth,(latparams[:,1]-b0)/b0,'o-', c='r')
axl.plot(list_depth,(latparams[:,2]-c0)/c0,'o-', c='g')
axl.set_xlabel('normal to surface sample depth (µm)')
axl.set_ylabel('(lattice parameter - ref value)/ref value')
axl.axhline(0.001, ls='-.')
axl.axhline(-0.001, ls='-.')
axl.grid(True)

if 1:  # angles
    figl, axl= plt.subplots()
    axl.plot(list_depth,(latparams[:,3]-alpha0)/alpha0,'o-', c='r')
    axl.plot(list_depth,(latparams[:,4]-beta0)/beta0,'o-', c='g')
    axl.plot(list_depth,(latparams[:,5]-gamma0)/gamma0,'o-', c='b')
    axl.set_xlabel('normal to surface sample depth (µm)')
    axl.set_ylabel('(angle - ref value)/ref value')
    axl.axhline(0.001, ls='-.')
    axl.axhline(-0.001, ls='-.')
    axl.grid(True)


# study for different orientation and zero strain (textbook lattice parameters)

In [ ]:
# simulation
nbUBs=200
Rs = R.random(nbUBs,random_state=6548)
quatlist = Rs.as_quat()
UBlist = Rs.as_matrix()

In [ ]:
from tqdm import tqdm

allstrains = []
allUBs = []
alllatparams = []
for mat_idx in tqdm(range(len(UBlist)), desc='indexing and refining...'):

    UBmatrix = UBlist[mat_idx]
    grain = CP.Prepare_Grain(key_material, UBmatrix)

    sample_depth=0

    detectorparameterscorrected = copy.copy(detectorparameters)


    s_tth, s_chi, s_miller_ind, s_posx, s_posy, s_E= LT.SimulateLaue_full_np(grain, 5,22,detectorparameters,
                                                                             pixelsize=pixelsize,detectordiameter=300,
                                                                             dim=framedim,
                                                                             depth=10.,
                                                                            removeharmonics=1)
    # writing .cor file
    folder = '/home/micha/LaueProjects/DepthEffects'
    prefixcorfilename = 'testZrO2_%d'%mat_idx
    IOLT.writefile_cor(prefixcorfilename,s_tth, s_chi,s_posx, s_posy,1000./s_E, param=CCDCalibdict,
                      dirname_output = folder)

    corfilename = prefixcorfilename + '.cor'
    fullpathcorfile =os.path.join(folder,corfilename)
    # indexing 
    DataSet = ISS.spotsset()

    DataSet.importdatafromfile(fullpathcorfile)
    dict_loop = {'MATCHINGRATE_THRESHOLD_IAL': 100,
                   'MATCHINGRATE_ANGLE_TOL': 0.2,
                   'NBMAXPROBED': 6,
                   'central spots indices': [0,],
                   'AngleTolLUT': 0.5,
                   'UseIntensityWeights': False,
                   'nbSpotsToIndex':10000,
                   'list matching tol angles':[0.2,0.2,0.2,0.2],
                   'nlutmax':4,
                   'MinimumNumberMatches': 3,
                   'MinimumMatchingRate':3
                   }
    grainindex=0
    DataSet = ISS.spotsset()

    DataSet.pixelsize = CCDCalibdict['xpixelsize']
    DataSet.dim = CCDCalibdict['framedim']
    DataSet.detectordiameter = CCDCalibdict['detectordiameter']
    DataSet.kf_direction = CCDCalibdict['kf_direction']
    DataSet.key_material = key_material
    DataSet.emin = 5
    DataSet.emax = 22
    
    CheckFirstThisMatrix=UBmatrix
    previousResults = 1, [CheckFirstThisMatrix], 50, 50
    
    initCCDCalibdict = copy.copy(CCDCalibdict)

    sols=[]
    UBrefinedlist=[]
    latticesparamslist=[]
    list_depth = [-50,]#-25,-20,-15,-10,-5,0,5,10,15,20,25,50]
    for dd in list_depth:
        grainindex=0
        DataSet = ISS.spotsset()
        DataSet.importdatafromfile(fullpathcorfile)
        DataSet.pixelsize = CCDCalibdict['xpixelsize']
        DataSet.dim = CCDCalibdict['framedim']
        DataSet.detectordiameter = CCDCalibdict['detectordiameter']
        DataSet.kf_direction = CCDCalibdict['kf_direction']
        DataSet.key_material = key_material
        DataSet.emin = 5
        DataSet.emax = 22
        #deltaycen = dd/1000./DataSet.pixelsize
        #DataSet.CCDcalibdict['ycen']=initCCDCalibdict['ycen']+deltaycen
        #DataSet.detectorparameters[2]=DataSet.CCDcalibdict['ycen']

        DataSet.IndexSpotsSet(fullpathcorfile, key_material, 5, 22, dict_loop, None,
                                 use_file=1, # if 1, reimport data from file and reset also spots properties dictionary
                                 IMM=False,LUT=None,n_LUT=dict_loop['nlutmax'],
                              angletol_list=dict_loop['list matching tol angles'],
                                nbGrainstoFind=1,
                              starting_grainindex=0,
                              MatchingRate_List=[1, 1, 1,1,1,1,1,1],
                                verbose=0, previousResults=previousResults,
                              corfilename=corfilename,
                             depth=dd)

        UBmatrixrefined = DataSet.dict_grain_matrix[0]
        
        lattice_parameter_direct_strain = CP.computeLatticeParameters_from_UB(
                                                            UBmatrixrefined, key_material, 'a',
                                                            dictmaterials=dictLT.dict_Materials)

        print('lattice_parameter_direct_strain',lattice_parameter_direct_strain)
        latticesparamslist.append(lattice_parameter_direct_strain)
        
        UBrefinedlist.append(UBmatrixrefined)
        sols.append(np.ravel(DataSet.dict_grain_devstrain[0]))

    epsstrain=np.array(sols)
    UBs = np.array(UBrefinedlist)
    latparams = np.array(latticesparamslist)
    
    allstrains.append(epsstrain)
    allUBs.append(UBs)
    alllatparams.append(latparams)

    


In [ ]:
len(alllatparams), alllatparams, allstrains, allUBs

In [ ]:
import pickle
restuple = alllatparams, allstrains, allUBs

with open(os.path.join(folder, 'multiUBstudy_depthimpact_on_strain.pickle'),'wb') as f:
    pickle.dump(restuple, f)

In [ ]:
ar_devstrain= np.array(allstrains)[:,0]

In [ ]:
xx, xy, xz, _, yy, yz, _,_,zz = ar_devstrain.T

In [ ]:
ar_lat = np.array(alllatparams)[:,0]

In [ ]:
a,b,c, alpha, beta, gamma = ar_lat.T

In [ ]:
alpha

In [ ]:
counts, bins = np.histogram(b)
fig2, ax2= plt.subplots()
ax2.hist(bins[:-1], bins, weights=counts)
print(np.amin(b))

In [ ]:
np.mean(b), b0, np.std(b), 2*np.std(b)/b0

In [ ]:
counts, bins = np.histogram(c)
fig3, ax3= plt.subplots()
ax3.hist(bins[:-1], bins, weights=counts)


In [ ]:
np.mean(c), c0, np.std(c), 2*np.std(c)/c0

In [ ]:
counts, bins = np.histogram(alpha)
fig4, ax4= plt.subplots()
ax4.hist(bins[:-1], bins, weights=counts)


In [ ]:
np.mean(alpha), alpha0, np.std(alpha), 2*np.std(alpha)/alpha0

In [ ]:
counts, bins = np.histogram(beta)
fig4, ax4= plt.subplots()
ax4.hist(bins[:-1], bins, weights=counts)

In [ ]:
np.mean(beta), beta0, np.std(beta), 2*np.std(beta)/beta0

In [ ]:
counts, bins = np.histogram(gamma)
fig4, ax4= plt.subplots()
ax4.hist(bins[:-1], bins, weights=counts)

In [ ]:
np.mean(gamma), gamma0, np.std(gamma), 2*np.std(gamma)/gamma0

In [ ]:
## strain distribution

In [ ]:
dicostrain={0: [xx,'xx'], 1:[yy,'yy'],2:[zz,'zz'],3:[yz,'yz'],4:[xz,'xz'],5:[xy,'xy']}
for k in range(6):

    strain, strainname = dicostrain[k]

    counts, bins = np.histogram(strain)
    fig4, ax4= plt.subplots()
    ax4.hist(bins[:-1], bins, weights=counts)
    plt.savefig(os.path.join(folder,'straindistribution_%s'%strainname))

In [ ]:
np.amin(strain), np.amax(strain)